## Load useful libraries

In [1]:
import os
import shutil
from pathlib import Path

In [2]:
from biological_graph_database.config import ftp_root_ncbi
from biological_graph_database.tools import download_file
from biological_graph_database.tools import compute_file_md5, read_md5_file

## User settings

In [3]:
remote_directory = '/pub/taxonomy'
email_address = os.environ['EMAIL_ADDRESS']
directory_download_to = os.environ['BIOLOGICAL_GRAPH_DATABASE_HOME'] + '/data/taxonomy'

## Create the data directory for the taxonomy download

In [4]:
shutil.rmtree(directory_download_to, ignore_errors = True)
Path(directory_download_to).mkdir()

## Download the taxonomy data

In [5]:
file_to_process = download_file('http://' + ftp_root_ncbi + '/' + remote_directory + '/taxdump.tar.gz', directory_download_to)
file_to_process_md5 = download_file('http://' + ftp_root_ncbi + '/' + remote_directory + '/taxdump.tar.gz.md5', directory_download_to)

## Check whether download succeeded

In [6]:
assert compute_file_md5(file_to_process) == read_md5_file(file_to_process_md5)

## Decompress the downloaded file

In [7]:
directory_working = os.getcwd()
os.chdir(directory_download_to)
os.system('gunzip taxdump.tar.gz')
os.system('tar -xf taxdump.tar')
os.system('gzip taxdump.tar')
os.chdir(directory_working)

In [80]:
import re

with open(directory_download_to + '/readme.txt') as f:
    list_lines = [x.strip() for x in f.readlines()]

new_list_lines = []
for line in list_lines:
    new_line = line
    
    #new_line = line.split('\t')[0].split('--')[0].strip()
    #new_line = line.split('\t')[0].strip()
    
    new_line = re.sub(' +', ' ', new_line)
    new_list_lines.append(new_line)
list_lines = new_list_lines



dict_results = {}
current_dmp_filename = None
for line in list_lines:
    if line.find('.dmp') >= 0 and line.find('--') == -1:

        current_dmp_filename = line.split('.dmp')[0].strip() + '.dmp'
        
        if current_dmp_filename.find('(') >= 0:
            thing = current_dmp_filename.split('(')
            for item in thing:
                if '.dmp' in item:
                    current_dmp_filename = item.strip()

        current_dmp_filename = current_dmp_filename.replace(')', '')
        current_dmp_filename = current_dmp_filename.strip()
        
        dict_results[current_dmp_filename] = []
    else:
        if line.strip() != '':
            to_add = line

            if to_add.find('--') >= 0 or to_add.strip().find('comments') == 0:
                to_add = to_add.split('\t')[0]

                to_add = to_add.split('--')[0]
            
                to_add = re.sub(' +', ' ', to_add)
            
                to_add = to_add.replace('(0 or 1)', '')
                to_add = to_add.replace('(1 or 0)', '')

                to_add = to_add.strip()
                to_add = to_add.replace(' ', '_')


                
                to_add = to_add.strip()
                if current_dmp_filename != None and to_add != '':
                    dict_results[current_dmp_filename].append(to_add)


del(dict_results['*.dmp'])

pp.pprint(dict_results)

{'citations.dmp': ['cit_id',
                   'cit_key',
                   'pubmed_id',
                   'medline_id',
                   'url',
                   'text',
                   'taxid_list'],
 'delnodes.dmp': ['tax_id'],
 'division.dmp': ['division_id', 'division_cde', 'division_name', 'comments'],
 'gencode.dmp': ['genetic_code_id', 'abbreviation', 'name', 'cde', 'starts'],
 'images.dmp': ['image_id',
                'image_key',
                'url',
                'license',
                'attribution',
                'source',
                'properties',
                'taxid_list'],
 'merged.dmp': ['old_tax_id', 'new_tax_id'],
 'names.dmp': ['tax_id', 'name_txt', 'unique_name', 'name_class'],
 'nodes.dmp': ['tax_id',
               'parent_tax_id',
               'rank',
               'embl_code',
               'division_id',
               'inherited_div_flag',
               'genetic_code_id',
               'inherited_GC_flag',
               'mitoc